In [1]:
import pandas as pd

df = pd.read_csv('hh_cleaned.csv')
print(df.columns.tolist())

['gender_age', 'salary', 'job_title', 'city_relocate', 'employment', 'schedule', 'work_experience', 'last_job', 'last_position', 'education', 'update_date', 'car', 'age', 'city', 'city_group']


In [2]:
import re

def extract_years(exp):
    if pd.isna(exp):
        return 0
    years = re.search(r'(\d+)\s*год', exp)
    months = re.search(r'(\d+)\s*месяц', exp)
    y = int(years.group(1)) if years else 0
    m = int(months.group(1)) if months else 0
    return round(y + m/12, 1)

df['exp_years'] = df['work_experience'].apply(extract_years)
print(df['exp_years'].describe())

count    44744.000000
mean        23.480190
std        201.187838
min          0.000000
25%          1.300000
50%          2.200000
75%          3.600000
max       2021.800000
Name: exp_years, dtype: float64


In [3]:
df['exp_years'] = df['exp_years'].clip(upper=50)
print(df['exp_years'].describe())

count    44744.000000
mean         3.613539
std          6.518584
min          0.000000
25%          1.300000
50%          2.200000
75%          3.600000
max         50.000000
Name: exp_years, dtype: float64


In [4]:
# Уникальные значения
print(df['employment'].value_counts())
print(df['schedule'].value_counts())

employment
полная занятость                                                                     30026
частичная занятость, полная занятость                                                 4835
частичная занятость, проектная работа, полная занятость                               3980
проектная работа, полная занятость                                                     985
стажировка, частичная занятость, проектная работа, полная занятость                    890
проектная работа, частичная занятость, полная занятость                                782
частичная занятость                                                                    630
стажировка, частичная занятость, полная занятость                                      613
частичная занятость, проектная работа                                                  494
стажировка, полная занятость                                                           444
стажировка, волонтерство, частичная занятость, проектная работа, полная занятос

In [5]:
# Для занятости
employment_types = ['полная занятость', 'частичная занятость', 'проектная работа', 'стажировка']
for emp in employment_types:
    df[f'emp_{emp}'] = df['employment'].str.contains(emp).astype(int)

# Для графика (аналогично)
schedule_types = ['полный день', 'удаленная работа', 'гибкий график', 'сменный график']
for sch in schedule_types:
    df[f'sch_{sch}'] = df['schedule'].str.contains(sch).astype(int)

print(df[[f'emp_{e}' for e in employment_types]].sum())

emp_полная занятость       43284
emp_частичная занятость    13136
emp_проектная работа        8068
emp_стажировка              2804
dtype: int64


In [6]:
df.to_csv('hh_cleaned_final.csv', index=False)

In [2]:
import pandas as pd
df = pd.read_csv('hh_cleaned_final.csv')
print(df.isna().sum())

gender_age                    0
salary                     2273
job_title                     0
city_relocate                 0
employment                    0
schedule                      0
work_experience             168
last_job                      1
last_position                 2
education                     0
update_date                   0
car                           0
age                           0
city                          0
city_group                    0
exp_years                     0
emp_полная занятость          0
emp_частичная занятость       0
emp_проектная работа          0
emp_стажировка                0
sch_полный день               0
sch_удаленная работа          0
sch_гибкий график             0
sch_сменный график            0
dtype: int64


In [3]:
df.isna().sum()

gender_age                    0
salary                     2273
job_title                     0
city_relocate                 0
employment                    0
schedule                      0
work_experience             168
last_job                      1
last_position                 2
education                     0
update_date                   0
car                           0
age                           0
city                          0
city_group                    0
exp_years                     0
emp_полная занятость          0
emp_частичная занятость       0
emp_проектная работа          0
emp_стажировка                0
sch_полный день               0
sch_удаленная работа          0
sch_гибкий график             0
sch_сменный график            0
dtype: int64

In [4]:
# Уникальные значения в опыте работы
print(df['work_experience'].nunique())

# Самая распространённая должность
print(df['job_title'].value_counts().head(1))

44413
job_title
Системный администратор    3099
Name: count, dtype: int64


In [5]:
def get_edu_level(text):
    if pd.isna(text):
        return None
    first_words = str(text).split()[:3]
    edu_text = ' '.join(first_words).lower()
    
    if 'высшее' in edu_text:
        return 'высшее'
    elif 'неоконченное высшее' in edu_text:
        return 'неоконченное высшее'
    elif 'среднее специальное' in edu_text:
        return 'среднее специальное'
    elif 'среднее' in edu_text:
        return 'среднее'
    else:
        return 'другое'

df['Образование'] = df['education'].apply(get_edu_level)

# Проверка
print(df['Образование'].value_counts())

# Ответ на вопрос (среднее = школьное)
print(df[df['Образование'] == 'среднее'].shape[0])

Образование
высшее                 38420
среднее специальное     5765
среднее                  559
Name: count, dtype: int64
559


In [6]:
# Извлечение пола и возраста
df['Пол'] = df['gender_age'].str.split(' , ').str[0]
df['Возраст'] = df['gender_age'].str.extract(r'(\d+)').astype(float)

# Преобразование пола в М/Ж
df['Пол'] = df['Пол'].map({'Мужчина': 'М', 'Женщина': 'Ж'})

# Проверка
print(df['Пол'].value_counts(normalize=True) * 100)
print(df['Возраст'].mean())

Пол
М    80.929287
Ж    19.070713
Name: proportion, dtype: float64
32.19674146254246


In [8]:
import re

def exp_to_months(exp):
    if pd.isna(exp) or exp == 'Не указано':
        return None
    years = 0
    months = 0
    year_match = re.search(r'(\d+)\s*год', exp)
    if year_match:
        years = int(year_match.group(1))
    month_match = re.search(r'(\d+)\s*месяц', exp)
    if month_match:
        months = int(month_match.group(1))
    return years * 12 + months

df['Опыт работы (месяц)'] = df['work_experience'].apply(exp_to_months)
print(df['Опыт работы (месяц)'].median())

27.0


In [9]:
print(df['Опыт работы (месяц)'].describe())

count    44574.000000
mean       282.835106
std       2418.790425
min          0.000000
25%         16.000000
50%         27.000000
75%         43.000000
max      24262.000000
Name: Опыт работы (месяц), dtype: float64


In [10]:
print(df['Опыт работы (месяц)'].value_counts().head(20))

Опыт работы (месяц)
20.0    1313
21.0    1292
17.0    1220
22.0    1214
19.0    1214
23.0    1206
15.0    1200
16.0    1192
14.0    1132
13.0    1122
18.0    1107
33.0     904
32.0     892
29.0     884
28.0     875
25.0     840
27.0     805
34.0     804
26.0     778
35.0     765
Name: count, dtype: int64


In [11]:
print(df['Опыт работы (месяц)'].isna().sum())

170


In [12]:
clean_exp = df[df['work_experience'] != 'Не указано']['Опыт работы (месяц)']
print(clean_exp.median())

27.0


In [13]:
million_cities = ['Новосибирск', 'Екатеринбург', 'Нижний Новгород', 'Казань', 'Челябинск', 
                   'Омск', 'Самара', 'Ростов-на-Дону', 'Уфа', 'Красноярск', 'Пермь', 
                   'Воронеж', 'Волгоград']

def extract_city(text):
    if pd.isna(text):
        return 'другие'
    city = text.split(',')[0].strip()
    if city == 'Москва':
        return 'Москва'
    elif city == 'Санкт-Петербург':
        return 'Санкт-Петербург'
    elif city in million_cities:
        return 'город-миллионник'
    else:
        return 'другие'

def extract_relocate(text):
    if pd.isna(text):
        return False
    return 'переезд' in text.lower() and 'не' not in text.lower().split('переезд')[0]

def extract_travel(text):
    if pd.isna(text):
        return False
    travel_text = text.lower()
    if 'командировк' in travel_text:
        if 'не готов' in travel_text or 'не готова' in travel_text:
            return False
        else:
            return True
    return False

df['Город'] = df['city_relocate'].apply(extract_city)
df['Готовность к переезду'] = df['city_relocate'].apply(extract_relocate)
df['Готовность к командировкам'] = df['city_relocate'].apply(extract_travel)

# Проценты
print("Санкт-Петербург: ", (df['Город'] == 'Санкт-Петербург').mean() * 100)
print("Готовы и к переезду, и к командировкам: ", ((df['Готовность к переезду']) & (df['Готовность к командировкам'])).mean() * 100)

Санкт-Петербург:  11.033881637761487
Готовы и к переезду, и к командировкам:  29.380475594493117


In [14]:
print(df['Готовность к командировкам'].value_counts(normalize=True))

Готовность к командировкам
False    0.681343
True     0.318657
Name: proportion, dtype: float64
